# Notebook 4 - APIM Specialist Exposure and Product Split

This notebook exposes Product Finder specialist agents behind APIM with a contract-first workflow.

## Sequence
1. Load environment and specialist registry
2. Create/update APIM backend and dynamic API route
3. Define contract (product, subscription, policy, naming)
4. Persist keys/outputs
5. Validate dynamic routing

In [ ]:
import json
import pathlib
import re
import subprocess
import sys

def find_repo_root(start: pathlib.Path) -> pathlib.Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / 'shared' / 'utils.py').exists() and (candidate / 'workshop' / 'product-finder').exists():
            return candidate
    raise RuntimeError('Could not locate repo root containing shared/utils.py and workshop/product-finder.')

repo_root = find_repo_root(pathlib.Path.cwd())
sys.path.insert(0, str(repo_root / 'shared'))
import utils  # type: ignore

def run(cmd: str, ok_msg: str = '', fail_msg: str = ''):
    return utils.run(cmd, ok_msg, fail_msg)

def run_json(cmd: str, ok_msg: str = '', fail_msg: str = ''):
    out = run(cmd, ok_msg, fail_msg)
    if not out.success:
        raise RuntimeError((out.text or '').strip() or fail_msg or f'Command failed: {cmd}')
    return out.json_data

def run_text(cmd: str, ok_msg: str = '', fail_msg: str = '') -> str:
    out = run(cmd, ok_msg, fail_msg)
    if not out.success:
        raise RuntimeError((out.text or '').strip() or fail_msg or f'Command failed: {cmd}')
    return (out.text or '').strip()

def azd_get_value(key: str) -> str:
    p = subprocess.run(['azd', 'env', 'get-value', key], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f'Missing azd env value [{key}]: {(p.stderr or p.stdout).strip()}')
    value = (p.stdout or '').strip()
    if not value or value.upper().startswith('ERROR:'):
        raise RuntimeError(f'Missing azd env value [{key}]')
    return value

def azd_get_optional(key: str, default: str = '') -> str:
    p = subprocess.run(['azd', 'env', 'get-value', key], capture_output=True, text=True)
    if p.returncode != 0:
        return default
    value = (p.stdout or '').strip()
    if not value or value.upper().startswith('ERROR:'):
        return default
    return value

def azd_set_value(key: str, value: str):
    p = subprocess.run(['azd', 'env', 'set', key, str(value)], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f'Failed to set azd env {key}: {(p.stderr or p.stdout).strip()}')

## 1. Load environment and specialist registry

In [ ]:
sub_id = azd_get_value('AZURE_SUBSCRIPTION_ID')
apim_rg = azd_get_value('PF_HUB_RG')
apim_name = azd_get_value('PF_APIM_NAME')

keyvault_name = azd_get_optional('PF_KEYVAULT_NAME', azd_get_optional('SPOKE_KEY_VAULT_NAME', ''))
keyvault_rg = azd_get_optional('PF_KEYVAULT_RG', azd_get_optional('SPOKE_RESOURCE_GROUP', ''))
if not keyvault_name:
    raise RuntimeError('Missing Key Vault name. Run Notebook 1 first to persist PF_KEYVAULT_NAME.')

kv_cmd = f'az keyvault show -n "{keyvault_name}"'
if keyvault_rg:
    kv_cmd += f' -g "{keyvault_rg}"'
kv_cmd += ' -o json'
run_json(kv_cmd, 'Key Vault lookup OK', 'Key Vault lookup failed')

deployed_specialists_raw = azd_get_optional('PF_DEPLOYED_SPECIALISTS', '').strip()
if not deployed_specialists_raw:
    raise RuntimeError(
        'Missing PF_DEPLOYED_SPECIALISTS in azd env. '
        'Run Notebook 3 first so deployed specialists are persisted.'
    )

try:
    deployed_specialists = json.loads(deployed_specialists_raw)
except Exception as ex:
    raise RuntimeError(
        f'Invalid PF_DEPLOYED_SPECIALISTS JSON in azd env: {ex}. '
        'Re-run Notebook 3 to repopulate it.'
    ) from ex

if not isinstance(deployed_specialists, list):
    raise RuntimeError(
        'PF_DEPLOYED_SPECIALISTS must be a JSON list of specialist names. '
        'Re-run Notebook 3 to repopulate it.'
    )

specialists = []
for name in deployed_specialists:
    name = str(name or '').strip()
    if not name or name == 'pf-orchestrator':
        continue
    specialists.append({'agent_name': name})

if not specialists:
    raise RuntimeError(
        'PF_DEPLOYED_SPECIALISTS is present but no specialist names were found. '
        'Re-run Notebook 3 and ensure specialist deployment succeeded.'
    )

utils.print_info(f'APIM: {apim_name} in {apim_rg}')
utils.print_info(f'Key Vault: {keyvault_name}' + (f' (RG: {keyvault_rg})' if keyvault_rg else ''))
utils.print_info(f'Specialists detected: {[a.get("agent_name") for a in specialists]}')

## 2. Create/update APIM backend and dynamic API

Create or update one APIM backend for the Foundry project and one dynamic API route.
The route `/agents/{agent}/invoke` is reused for all specialists; policy resolves `{agent}` to the target runtime path.

In [ ]:
import requests
from apimtools import APIMClientTool  # type: ignore[reportMissingImports]

# Reuse the same APIM helper pattern used in other notebooks.
apimClientTool = APIMClientTool(apim_rg, apim_name)
apimClientTool.initialize()

apim_gateway_url = azd_get_optional('APIM_GATEWAY_URL', '').rstrip('/')
if not apim_gateway_url:
    apim_gateway_url = str(getattr(apimClientTool, 'apim_resource_gateway_url', '') or '').rstrip('/')
if not apim_gateway_url:
    apim_gateway_url = run_text(
        f'az apim show -g "{apim_rg}" -n "{apim_name}" --query gatewayUrl -o tsv',
        ok_msg='Resolved APIM gateway URL from APIM resource',
        fail_msg='Failed to resolve APIM gateway URL',
    ).rstrip('/')
if not apim_gateway_url:
    raise RuntimeError('APIM gateway URL is empty. Set APIM_GATEWAY_URL or verify APIM gateway provisioning.')

apim_identity_suffix = apim_name[5:] if apim_name.startswith('apim-') else apim_name
specialized_agents_identity_name = azd_get_optional(
    'PF_APIM_MANAGED_IDENTITY_NAME',
    f'id-specialized-agents-pf-{apim_identity_suffix}',
).strip().lower()
utils.print_info(f'Specialized-agents managed identity target: {specialized_agents_identity_name}')

foundry_account = azd_get_value('SPOKE_AI_FOUNDRY_ACCOUNT_NAME')
foundry_project = azd_get_value('SPOKE_AI_FOUNDRY_PROJECT_NAME')
foundry_project_endpoint = f'https://{foundry_account}.services.ai.azure.com/api/projects/{foundry_project}'

backend_identity_show = run(
    f'az identity show --resource-group "{apim_rg}" --name "{specialized_agents_identity_name}" -o json'
)
if not backend_identity_show.success:
    backend_identity = run_json(
        f'az identity create --resource-group "{apim_rg}" --name "{specialized_agents_identity_name}" -o json',
        ok_msg='Created specialized-agents managed identity',
        fail_msg='Failed to create specialized-agents managed identity',
    )
else:
    backend_identity = backend_identity_show.json_data
    utils.print_info(f'Specialized-agents managed identity already exists: {specialized_agents_identity_name}')

backend_uami_name = str(backend_identity.get('name') or specialized_agents_identity_name).strip()
backend_uami_resource_id = str(backend_identity.get('id') or '').strip()
backend_uami_client_id = str(backend_identity.get('clientId') or '').strip()
backend_uami_principal_id = str(backend_identity.get('principalId') or '').strip()

if not backend_uami_resource_id or not backend_uami_client_id or not backend_uami_principal_id:
    raise RuntimeError('Could not resolve specialized-agents managed identity resourceId/clientId/principalId.')

utils.print_ok(f'Found user-assigned managed identity: {backend_uami_name}')
utils.print_info(f'Client ID: {backend_uami_client_id}')
utils.print_ok(f'Managed Identity Name: {backend_uami_name}')

# Get ARM token using the existing helper (works with this environment's shell configuration).
token_json = run_json(
    'az account get-access-token --resource https://management.azure.com/ -o json',
    ok_msg='Retrieved ARM access token',
    fail_msg='Failed to get ARM access token',
)
access_token = str(token_json.get('accessToken', '')).strip()
if not access_token:
    raise RuntimeError('ARM access token is empty.')

apim_service_url = (
    f'https://management.azure.com/subscriptions/{sub_id}'
    f'/resourceGroups/{apim_rg}'
    f'/providers/Microsoft.ApiManagement/service/{apim_name}?api-version=2024-05-01'
)
apim_identity = run_json(
    f'az apim show --resource-group {apim_rg} --name {apim_name} --query identity -o json',
    ok_msg='Resolved APIM identity for backend auth',
    fail_msg='Failed to resolve APIM identity for backend auth',
)
existing_uamis = {}
apim_identity_type = 'UserAssigned'
if isinstance(apim_identity, dict):
    apim_identity_type = str(apim_identity.get('type') or 'UserAssigned').strip() or 'UserAssigned'
    current_uamis = apim_identity.get('userAssignedIdentities') or {}
    if isinstance(current_uamis, dict):
        existing_uamis = {rid: {} for rid in current_uamis.keys() if str(rid).strip()}

if backend_uami_resource_id not in existing_uamis:
    existing_uamis[backend_uami_resource_id] = {}
    if 'UserAssigned' not in apim_identity_type:
        apim_identity_type = f'{apim_identity_type},UserAssigned' if apim_identity_type else 'UserAssigned'

    apim_identity_payload = {
        'identity': {
            'type': apim_identity_type,
            'userAssignedIdentities': existing_uamis,
        }
    }
    apim_identity_resp = requests.patch(
        apim_service_url,
        headers={
            'Authorization': f'Bearer {access_token}',
            'Content-Type': 'application/json',
        },
        json=apim_identity_payload,
        timeout=90,
    )
    if apim_identity_resp.status_code >= 400:
        raise RuntimeError(
            'Failed to attach specialized-agents managed identity to APIM. '
            f'HTTP {apim_identity_resp.status_code}. Response: {(apim_identity_resp.text or "")[:1200]}'
        )
    utils.print_ok(f'Attached {backend_uami_name} to APIM')
else:
    utils.print_info(f'{backend_uami_name} is already attached to APIM')

backend_id = 'pf-foundry-project-backend'
backend_url = (
    f'https://management.azure.com/subscriptions/{sub_id}'
    f'/resourceGroups/{apim_rg}'
    f'/providers/Microsoft.ApiManagement/service/{apim_name}'
    f'/backends/{backend_id}?api-version=2024-05-01'
)
backend_payload = {
    'properties': {
        'url': foundry_project_endpoint,
        'protocol': 'http',
        'title': 'PF Foundry Project Backend',
        'credentials': {
            'managedIdentity': {
                'clientId': backend_uami_client_id,
                'resource': 'https://ai.azure.com',
            }
        },
    }
}

backend_resp = requests.put(
    backend_url,
    headers={
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json',
    },
    json=backend_payload,
    timeout=90,
)
if backend_resp.status_code >= 400:
    raise RuntimeError(
        'Failed to create/update APIM backend for Foundry project. '
        f'HTTP {backend_resp.status_code}. Response: {(backend_resp.text or "")[:1200]}'
    )
utils.print_ok('Created or updated APIM backend for Foundry project')

api_id = 'pf-agents-api'
api_path = 'agents'
api_display_name = 'pf-agents'
api_show = run(
    f'az apim api show --resource-group "{apim_rg}" --service-name "{apim_name}" --api-id "{api_id}" -o json'
)
if not api_show.success:
    run_json(
        f'az apim api create --resource-group "{apim_rg}" --service-name "{apim_name}" --api-id "{api_id}" --display-name "{api_display_name}" --path "{api_path}" --protocols https --service-url "{foundry_project_endpoint}" -o json',
        ok_msg='Created dynamic APIM API for all specialists',
        fail_msg='Failed to create dynamic APIM API for all specialists',
    )
else:
    run_json(
        f'az apim api update --resource-group "{apim_rg}" --service-name "{apim_name}" --api-id "{api_id}" --set serviceUrl="{foundry_project_endpoint}" -o json',
        ok_msg='Updated dynamic APIM API for all specialists',
        fail_msg='Failed to update dynamic APIM API for all specialists',
    )

op_id = 'invoke-agent'
operation_url = (
    f'https://management.azure.com/subscriptions/{sub_id}'
    f'/resourceGroups/{apim_rg}'
    f'/providers/Microsoft.ApiManagement/service/{apim_name}'
    f'/apis/{api_id}/operations/{op_id}?api-version=2024-05-01'
)
operation_payload = {
    'properties': {
        'displayName': 'Invoke agent',
        'method': 'POST',
        'urlTemplate': '/{agent}/invoke',
        'templateParameters': [
            {
                'name': 'agent',
                'description': 'Agent route key',
                'type': 'string',
                'required': True,
            }
        ],
    }
}
operation_resp = requests.put(
    operation_url,
    headers={
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json',
    },
    json=operation_payload,
    timeout=90,
)
if operation_resp.status_code >= 400:
    raise RuntimeError(
        'Failed to create/update dynamic invoke operation for all specialists. '
        f'HTTP {operation_resp.status_code}. Response: {(operation_resp.text or "")[:1200]}'
    )
utils.print_ok('Created or updated dynamic invoke operation for all specialists')

api_ids = [api_id]
runtime_by_agent = {}
for agent in specialists:
    name = str(agent.get('agent_name', '')).strip()
    if not name:
        continue
    route_suffix = re.sub(r'[^a-z0-9-]', '-', name.lower())
    runtime_by_agent[name] = f'{apim_gateway_url}/{api_path}/{route_suffix}/invoke'
    utils.print_ok(f'{name} runtime route ready: {runtime_by_agent[name]}')

utils.print_ok('Prepared single dynamic API for product attachment')

## 3. Define contract (product, subscription, policy, naming)

Create or update the internal APIM product using deterministic naming.
Apply product policy and ensure a deterministic subscription exists.
Retrieve the subscription key for downstream orchestrator access.

In [ ]:
import datetime
import os

AGENT_CONTRACT = {
    'service_code': 'AGT',
    'business_unit': 'PF',
    'use_case_name': 'ProductFinderAgentAccess',
    'environment': 'DEV',
    'endpoint_secret': 'PF_AGENT_ENDPOINT',
    'apikey_secret': 'PF_AGENT_KEY',
}

agent_product_id = (
    f"{AGENT_CONTRACT['service_code']}-"
    f"{AGENT_CONTRACT['business_unit']}-"
    f"{AGENT_CONTRACT['use_case_name']}-"
    f"{AGENT_CONTRACT['environment']}"
)
agent_product_display = 'PF Specialized Agent Access'
agent_subscription_name = f'{agent_product_id}-SUB-01'
agent_subscription_key_secret_name = 'PF_AGENT_SUBSCRIPTION_KEY'

agent_policy_xml = """<policies>
  <inbound>
    <base />
    <rate-limit-by-key calls="120" renewal-period="60" counter-key="@(context.Subscription.Id)" />
    <set-variable name="enableResponseHeaders" value="@(true)" />
    <set-variable name="agentName" value="@{
        var segments = context.Request.OriginalUrl.Path.Trim('/').Split('/');
        for (var i = 0; i < segments.Length - 1; i++)
        {
            if (string.Equals(segments[i], &quot;agents&quot;, StringComparison.OrdinalIgnoreCase))
            {
                return segments[i + 1];
            }
        }
        return string.Empty;
    }" />
    <choose>
      <when condition="@(!string.IsNullOrEmpty((string)context.Variables[&quot;agentName&quot;]))">
        <set-backend-service backend-id="pf-foundry-project-backend" />
        <rewrite-uri template="@(&quot;/agents/&quot; + (string)context.Variables[&quot;agentName&quot;] + &quot;/endpoint/protocols/openai/responses?api-version=2025-05-15-preview&quot;)" copy-unmatched-params="false" />
      </when>
      <otherwise>
        <return-response>
          <set-status code="400" reason="Bad Request" />
          <set-header name="Content-Type" exists-action="override">
            <value>application/json</value>
          </set-header>
          <set-body>{"error":"Missing agent name in path. Expected route containing /agents/{agentName}."}</set-body>
        </return-response>
      </otherwise>
    </choose>
  </inbound>
  <backend>
    <base />
  </backend>
  <outbound>
    <base />
  </outbound>
  <on-error>
    <base />
  </on-error>
</policies>"""

# Match Notebook 1 contract artifact flow: generate policy and bicepparam files.
template_file = repo_root / 'bicep' / 'infra' / 'citadel-access-contracts' / 'main.bicep'
pf_contracts_dir = repo_root / 'workshop' / 'product-finder' / 'access-contracts'
folder_name = f"{AGENT_CONTRACT['business_unit'].lower()}-{AGENT_CONTRACT['use_case_name'].lower()}"
env_folder = AGENT_CONTRACT['environment'].lower()
contract_folder = pf_contracts_dir / 'contracts' / folder_name / env_folder
contract_folder.mkdir(parents=True, exist_ok=True)
(contract_folder / 'ai-product-policy.xml').write_text(agent_policy_xml, encoding='utf-8')

using_path = pathlib.Path(os.path.relpath(template_file, contract_folder)).as_posix()
spoke_rg = azd_get_value('SPOKE_RESOURCE_GROUP')
foundry_account = azd_get_value('SPOKE_AI_FOUNDRY_ACCOUNT_NAME')
foundry_project = azd_get_value('SPOKE_AI_FOUNDRY_PROJECT_NAME')
agent_connection_prefix = azd_get_optional('PF_MODEL_CONNECTION', 'Product-Finder-DEV-LLM')
if agent_connection_prefix.endswith('-LLM'):
    agent_connection_prefix = agent_connection_prefix[:-4]

if not api_ids:
    raise RuntimeError('No APIs found to map in contract. Run Cell 6 (Create/update APIM backend and dynamic API) first.')

api_name_mapping_entries = ', '.join([f"'{api_id}'" for api_id in api_ids])

params_content = f"""using '{using_path}'

param apim = {{
  subscriptionId: '{sub_id}'
  resourceGroupName: '{apim_rg}'
  name: '{apim_name}'
}}

param keyVault = {{
  subscriptionId: '{sub_id}'
  resourceGroupName: '{keyvault_rg}'
  name: '{keyvault_name}'
}}

param useTargetAzureKeyVault = true

param useCase = {{
  businessUnit: '{AGENT_CONTRACT['business_unit']}'
  useCaseName: '{AGENT_CONTRACT['use_case_name']}'
  environment: '{AGENT_CONTRACT['environment']}'
}}

param apiNameMapping = {{
  {AGENT_CONTRACT['service_code']}: [{api_name_mapping_entries}]
}}

param services = [
  {{
    code: '{AGENT_CONTRACT['service_code']}'
    endpointSecretName: '{AGENT_CONTRACT['endpoint_secret']}'
    apiKeySecretName: '{AGENT_CONTRACT['apikey_secret']}'
    policyXml: loadTextContent('ai-product-policy.xml')
  }}
]

param productTerms = 'Product Finder specialist-agent access contract (generated by Notebook 4)'

param useTargetFoundry = true
param foundry = {{
  subscriptionId: '{sub_id}'
  resourceGroupName: '{spoke_rg}'
  accountName: '{foundry_account}'
  projectName: '{foundry_project}'
}}

param foundryConfig = {{
  connectionNamePrefix: '{agent_connection_prefix}'
  connectionCategory: 'ApiManagement'
  deploymentInPath: 'false'
  isSharedToAll: false
  inferenceAPIVersion: '2025-03-01-preview'
  deploymentAPIVersion: ''
  staticModels: []
  listModelsEndpoint: ''
  getModelEndpoint: ''
  deploymentProvider: ''
  customHeaders: {{}}
  authConfig: {{}}
}}
"""
params_file = contract_folder / 'main.bicepparam'
params_file.write_text(params_content, encoding='utf-8')
utils.print_info(f'Contract artifacts written to: {contract_folder}')

# Follow Notebook 1 pattern: deploy the Citadel access contract template and then read subscription secrets via APIMClientTool.
azure_location = azd_get_value('AZURE_LOCATION')
deployment_name = f"pf-agent-contract-{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d%H%M%S')}"
deploy_cmd = (
    f'az deployment sub create --name {deployment_name} '
    f'--location {azure_location} '
    f'--template-file "{template_file}" '
    f'--parameters "{params_file}" -o json'
)
run_json(
    deploy_cmd,
    ok_msg='Internal agent contract deployed',
    fail_msg='Internal agent contract deployment failed',
)

# Ensure the dedicated specialized-agents UAMI has the required Foundry data-plane roles.
uami_principal_id = str(globals().get('backend_uami_principal_id', '') or '').strip()
if not uami_principal_id:
    identity_name = azd_get_optional('PF_APIM_MANAGED_IDENTITY_NAME', '').strip()
    if not identity_name:
        apim_identity_suffix = apim_name[5:] if apim_name.startswith('apim-') else apim_name
        identity_name = f'id-specialized-agents-pf-{apim_identity_suffix}'.lower()
    identity_info = run_json(
        f'az identity show --resource-group {apim_rg} --name "{identity_name}" -o json',
        ok_msg='Resolved specialized-agents managed identity for RBAC',
        fail_msg='Failed to resolve specialized-agents managed identity for RBAC',
    )
    uami_principal_id = str(identity_info.get('principalId') or '').strip()
    backend_uami_name = str(identity_info.get('name') or identity_name).strip()
    backend_uami_client_id = str(identity_info.get('clientId') or '').strip()
    backend_uami_resource_id = str(identity_info.get('id') or '').strip()

if not uami_principal_id:
    raise RuntimeError('Could not resolve specialized-agents UAMI principalId for Foundry RBAC assignment.')

foundry_account_scope = (
    f'/subscriptions/{sub_id}/resourceGroups/{spoke_rg}'
    f'/providers/Microsoft.CognitiveServices/accounts/{foundry_account}'
)
foundry_project_scope = (
    f'{foundry_account_scope}'
    f'/projects/{foundry_project}'
)

existing_project_rbac = run_json(
    f"az role assignment list --assignee-object-id {uami_principal_id} "
    f"--scope {foundry_project_scope} --query \"[?roleDefinitionName=='Azure AI Developer']\" -o json",
    ok_msg='Checked specialized-agents Foundry project RBAC',
    fail_msg='Failed to check specialized-agents Foundry project RBAC',
)
if not existing_project_rbac:
    run_json(
        f"az role assignment create --assignee-object-id {uami_principal_id} "
        f"--assignee-principal-type ServicePrincipal --role \"Azure AI Developer\" "
        f"--scope {foundry_project_scope} -o json",
        ok_msg='Created specialized-agents Foundry project RBAC',
        fail_msg='Failed to create specialized-agents Foundry project RBAC',
    )
else:
    utils.print_info('Specialized-agents Foundry project RBAC already exists')

from apimtools import APIMClientTool  # type: ignore[reportMissingImports]

apimClientTool = APIMClientTool(apim_rg, apim_name)
apimClientTool.initialize()

agent_subscription_key = ''
for sub in apimClientTool.apim_subscriptions:
    if sub.get('name') == agent_subscription_name:
        agent_subscription_key = str(sub.get('key', '')).strip()
        break

if not agent_subscription_key:
    raise RuntimeError(
        f'Could not retrieve subscription key for {agent_subscription_name} after deployment. '
        'Check APIM deployment outputs and subscription provisioning in APIM.'
    )

utils.print_ok('Internal APIM agent contract is defined')

## 4. Persist keys/outputs

Persist gateway, product, subscription, and API outputs to azd env.
Store the APIM agent subscription key in Key Vault and persist secret references for downstream notebooks.

In [ ]:
azd_set_value('APIM_GATEWAY_URL', apim_gateway_url)
azd_set_value('PF_KEYVAULT_NAME', keyvault_name)
azd_set_value('PF_KEYVAULT_RG', keyvault_rg)
azd_set_value('PF_AGENT_APIM_PRODUCT_ID', agent_product_id)
azd_set_value('PF_AGENT_SUBSCRIPTION_NAME', agent_subscription_name)
azd_set_value('PF_AGENT_SUBSCRIPTION_KEY_SECRET_NAME', agent_subscription_key_secret_name)
azd_set_value('PF_APIM_MANAGED_IDENTITY_NAME', backend_uami_name)
azd_set_value('PF_APIM_MANAGED_IDENTITY_CLIENT_ID', backend_uami_client_id)
azd_set_value('PF_APIM_MANAGED_IDENTITY_PRINCIPAL_ID', backend_uami_principal_id)
azd_set_value('PF_APIM_MANAGED_IDENTITY_RESOURCE_ID', backend_uami_resource_id)
# Keep plain key for workshop backward compatibility; prefer Key Vault secret in downstream consumers.
azd_set_value('PF_AGENT_SUBSCRIPTION_KEY', agent_subscription_key)
azd_set_value('PF_AGENT_APIM_API_IDS', ','.join(api_ids))
azd_set_value('PF_AGENT_APIM_EXPOSURE_READY', 'true')

utils.print_ok('Notebook 4 complete. Continue with Notebook 5 for API Center sync.')

## 5. Validate dynamic APIM routing

Run a direct APIM call through `/agents/{agentName}/invoke` to confirm product policy path parsing and backend rewrite are working.

In [ ]:
import re
import uuid
import requests

if not specialists:
    raise RuntimeError('No specialists available. Run Cell 4 first to load registry and specialists.')

test_agent = str(specialists[0].get('agent_name', '')).strip()
if not test_agent:
    raise RuntimeError('First specialist has no agent_name in registry metadata.')

route_suffix = re.sub(r'[^a-z0-9-]', '-', test_agent.lower())
test_url = f"{apim_gateway_url.rstrip('/')}/agents/{route_suffix}/invoke"

if not agent_subscription_key:
    agent_subscription_key = azd_get_optional('PF_AGENT_SUBSCRIPTION_KEY', '').strip()
if not agent_subscription_key:
    raise RuntimeError('Missing agent subscription key. Run Cell 8 and Cell 12 first.')

active_identity_name = azd_get_optional('PF_APIM_MANAGED_IDENTITY_NAME', str(globals().get('backend_uami_name', '') or '')).strip()
active_identity_client_id = azd_get_optional('PF_APIM_MANAGED_IDENTITY_CLIENT_ID', str(globals().get('backend_uami_client_id', '') or '')).strip()
if active_identity_name:
    utils.print_info(f'Backend managed identity: {active_identity_name}')
if active_identity_client_id:
    utils.print_info(f'Backend managed identity client ID: {active_identity_client_id}')

headers = {
    'Ocp-Apim-Subscription-Key': agent_subscription_key,
    'Content-Type': 'application/json',
}
payload = {
    'input': f'Routing smoke test for {test_agent}. Reply with a short acknowledgement only.',
    'metadata': {
        'conversation_id': str(uuid.uuid4()),
        'routing_test': 'nb4-cell-14',
        'target_agent': test_agent,
    },
}

resp = requests.post(test_url, headers=headers, json=payload, timeout=90)
utils.print_info(f'Routing test URL: {test_url}')
utils.print_info(f'Routing test status: {resp.status_code}')

if resp.status_code >= 400:
    body = (resp.text or '')[:1200]
    backend_headers = {k.lower(): v for k, v in resp.headers.items()}

    # If Foundry headers are present, APIM routing succeeded and backend auth/RBAC is the failing layer.
    if resp.status_code == 403 and (
        'azureml-served-by-cluster' in backend_headers or 'x-ms-region' in backend_headers
    ):
        raise RuntimeError(
            'APIM reached Foundry backend, but backend authorization failed (HTTP 403). '
            'Ensure the specialized-agents identity has Cognitive Services User on the Foundry account '
            'and Azure AI Developer on the Foundry project, then retry after RBAC propagation. '
            f'Response snippet: {body}'
        )

    raise RuntimeError(
        'APIM routing test failed. '
        f'HTTP {resp.status_code}. Response snippet: {body}'
    )

try:
    response_json = resp.json()
except Exception:
    response_json = {'raw': (resp.text or '')[:1200]}

output_text = ''
if isinstance(response_json, dict):
    output_text = str(response_json.get('output_text') or '').strip()
    if not output_text:
        output = response_json.get('output')
        if isinstance(output, list):
            chunks = []
            for item in output:
                if not isinstance(item, dict):
                    continue
                for c in item.get('content', []):
                    if isinstance(c, dict) and c.get('text'):
                        chunks.append(str(c['text']))
            output_text = '\n'.join(chunks).strip()

if not output_text:
    output_text = str(response_json)[:600]

utils.print_ok(f'APIM dynamic routing test succeeded for {test_agent}')
utils.print_info(f'Response preview: {output_text[:300]}')